# ETL — WNM Excitatory: Projection Matrix

Writes two `ProjectionMeasurementMatrix` rows (ipsi + contra) for `project_id="visp_wnm"`, `dataset_id="visp_exc_wnm"`, plus the backing wide-form Delta tables. Source: `ProjectionMatrix_tip_and_branch_roll_up.csv` (345 cells × 152 ipsi + 68 contra regions). Prerequisite: `etl_wnm_exc_01`. Registers 4 cells absent from `_01` via `append_new_dataitems`.

**Caveat:** `measurement_type=MICRONS_OF_AXON` is a best guess. The filename `tip_and_branch_roll_up` suggests counts, but values are floats with magnitudes ~10⁴ — consistent with µm of axon length per region. To confirm with the data owner.

**Known schema mismatches (stopgaps):**

1. `ProjectionMeasurementMatrix` lacks `ProjectScoped` → metadata predicate is `id IN (...)` only. Fix: add `mixins: [ProjectScoped]` in `schemas/projection_schema.yaml` and regenerate.
2. `region_index` stores raw acronym strings instead of `BrainRegion.id`s; `brainregion/` is not yet populated. Re-run after that bootstrap.
3. `values` is typed `ZarrArray` but stored here as a `file://` delta-path string (mirrors `CellFeatureMatrix.parquet_path`). Fix: add a `parquet_path` slot or commit to zarr.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import pyarrow as pa
from deltalake import write_deltalake

from connects_common_connectivity.models import (
    DataItem,
    DataItemDataSetAssociation,
    Laterality,
    Modality,
    ProjectionMeasurementMatrix,
    ProjectionMeasurementType,
    Unit,
)
from connects_common_connectivity.config import output_root
from connects_common_connectivity.io import write_models, write_projection_matrix


In [2]:
INPUT_CSV   = "/data/exc_vis_manuscript_wnm_axon_projection/ProjectionMatrix_tip_and_branch_roll_up.csv"
OUTPUT_ROOT = output_root()
PROJECT_ID  = "visp_wnm"
DATASET_ID  = "visp_exc_wnm"

FSI_IPSI    = "wnm_exc_proj_ipsi"
FSI_CONTRA  = "wnm_exc_proj_contra"

print(f"INPUT_CSV   : {INPUT_CSV}")
print(f"OUTPUT_ROOT : {OUTPUT_ROOT}")
print(f"PROJECT_ID  : {PROJECT_ID}")
print(f"DATASET_ID  : {DATASET_ID}")
print(f"FSI_IPSI    : {FSI_IPSI}")
print(f"FSI_CONTRA  : {FSI_CONTRA}")

INPUT_CSV   : /data/exc_vis_manuscript_wnm_axon_projection/ProjectionMatrix_tip_and_branch_roll_up.csv
OUTPUT_ROOT : ../scratch/em_patchseq_wnm_v2/
PROJECT_ID  : visp_wnm
DATASET_ID  : visp_exc_wnm
FSI_IPSI    : wnm_exc_proj_ipsi
FSI_CONTRA  : wnm_exc_proj_contra


In [3]:
# Prerequisite: etl_wnm_exc_01 must have populated dataitem_dataset_association/ for this dataset.
prereq_assoc = (
    pl.read_delta(OUTPUT_ROOT + "dataitem_dataset_association/")
    .filter((pl.col("project_id") == PROJECT_ID) & (pl.col("dataset_id") == DATASET_ID))
)
assert prereq_assoc.shape[0] > 0, (
    f"etl_wnm_exc_01_dataset_dataitem.ipynb must be run first — "
    f"no DataItemDataSetAssociation rows for project_id='{PROJECT_ID}', dataset_id='{DATASET_ID}'"
)
print(f"Prereq OK: {prereq_assoc.shape[0]} DataItem associations registered for {DATASET_ID}.")

Prereq OK: 345 DataItem associations registered for visp_exc_wnm.


## Load projection matrix CSV

In [4]:
# First column (unnamed) is the swc filename. Strip the .swc suffix to get the cell id
# (matches etl_wnm_exc_01 convention). Cell ids are kept as strings — never cast.
df = pd.read_csv(INPUT_CSV, index_col=0)
df.index = df.index.astype(str).str.removesuffix(".swc")
df.index.name = "id"
print("Shape:", df.shape)
df.iloc[:3, :6]

Shape: (345, 220)


,ipsi_VISam,ipsi_VISp,ipsi_VISpm,ipsi_VISrl,contra_VISpor,ipsi_CP
id,,,,,,
18864_6734-X4899-Y27447_reg,8287.70664,34450.175934,483.223644,5737.760785,0.000000,0.000000
191812_7938-X6892-Y25312_reg,0.00000,794.102517,0.000000,0.000000,1045.437572,9243.339922
211550_7718-X19461-Y16950_reg,0.00000,6473.751624,0.000000,0.000000,0.000000,0.000000


In [5]:
# Split columns by prefix and strip the prefix to get region acronyms.
ipsi_cols   = [c for c in df.columns if c.startswith("ipsi_")]
contra_cols = [c for c in df.columns if c.startswith("contra_")]
other_cols  = [c for c in df.columns if not (c.startswith("ipsi_") or c.startswith("contra_"))]

ipsi_acronyms   = [c[len("ipsi_"):]   for c in ipsi_cols]
contra_acronyms = [c[len("contra_"):] for c in contra_cols]

print(f"num_ipsi={len(ipsi_cols)}  num_contra={len(contra_cols)}  other={len(other_cols)}  total={df.shape[1]}")
assert len(other_cols) == 0, f"Unexpected columns without ipsi_/contra_ prefix: {other_cols}"
assert len(ipsi_cols) + len(contra_cols) == df.shape[1], "Column counts do not sum to total"
# ipsi/contra acronyms may overlap (a region can appear on both sides), but column names must be unique.
assert len(set(df.columns)) == df.shape[1], "Duplicate column names in CSV"
print("Acronym overlap (region appears in both ipsi and contra):", len(set(ipsi_acronyms) & set(contra_acronyms)))

num_ipsi=152  num_contra=68  other=0  total=220
Acronym overlap (region appears in both ipsi and contra): 56


## Register new DataItems (cells in this CSV but not in `_01`)

In [6]:
cell_ids = df.index.tolist()
assert len(cell_ids) == len(set(cell_ids)), "Duplicate cell ids in CSV index"

registered_ids = set(prereq_assoc["dataitem_id"].to_list())
new_ids = [c for c in cell_ids if c not in registered_ids]
print(f"Cells in CSV         : {len(cell_ids)}")
print(f"Already registered   : {len(cell_ids) - len(new_ids)}")
print(f"New to register      : {len(new_ids)}")
if new_ids:
    print("New ids:", new_ids)

Cells in CSV         : 345
Already registered   : 345
New to register      : 0


In [7]:
# Append only-new DataItem rows for cells absent from _01. append_new_dataitems is idempotent —
# re-running this cell appends 0 and does not disturb other projects' rows in dataitem/.
if new_ids:
    new_items = [
        DataItem(id=cid, name=cid, project_id=PROJECT_ID, modality=Modality.MORPHOLOGY.value)
        for cid in new_ids
    ]
    n_appended = write_models(new_items, output_root=OUTPUT_ROOT).rows_written
    print(f"Appended {n_appended} new DataItem rows")
else:
    print("All cells already in DataItem; nothing to append.")

All cells already in DataItem; nothing to append.


In [8]:
# Verification: every cell in the CSV is now present in dataitem/ for this project.
di_verify = (
    pl.read_delta(OUTPUT_ROOT + "dataitem/")
    .filter(pl.col("project_id") == PROJECT_ID)
)
print(di_verify.shape)
print(di_verify.head(3))
registered_now = set(di_verify["id"].to_list())
missing = [c for c in cell_ids if c not in registered_now]
assert not missing, f"{len(missing)} cells from CSV are not registered: {missing[:5]}"
assert di_verify["id"].n_unique() == di_verify.shape[0], "Duplicate DataItem ids detected"
print("All", len(cell_ids), "cells present in DataItem.")

(345, 4)
shape: (3, 4)
┌───────────────────────────────┬───────────────────────────────┬───────────────────┬────────────┐
│ id                            ┆ name                          ┆ neuroglancer_link ┆ project_id │
│ ---                           ┆ ---                           ┆ ---               ┆ ---        │
│ str                           ┆ str                           ┆ str               ┆ str        │
╞═══════════════════════════════╪═══════════════════════════════╪═══════════════════╪════════════╡
│ 17109_6801-X7432-Y4405_reg    ┆ 17109_6801-X7432-Y4405_reg    ┆ null              ┆ visp_wnm   │
│ 211541_6961-X18505-Y15909_reg ┆ 211541_6961-X18505-Y15909_reg ┆ null              ┆ visp_wnm   │
│ 220309_5824-X3486-Y10261_reg  ┆ 220309_5824-X3486-Y10261_reg  ┆ null              ┆ visp_wnm   │
└───────────────────────────────┴───────────────────────────────┴───────────────────┴────────────┘
All 345 cells present in DataItem.


## Write `DataItemDataSetAssociation` as `existing ∪ 345 cells`

`DataItemDataSetAssociation` is `overwrite_scoped` on `(project_id, dataset_id)`, so passing
only the 345 cell ids from this CSV would clobber rows written by `_01`/`_02` for the same
scope. Union with the existing scope before re-writing.

In [9]:
# Re-assert the full (project_id, dataset_id) association scope as the union
# of any existing assoc rows and this CSV's cell_ids. DataItemDataSetAssociation
# is overwrite_scoped on (project_id, dataset_id), so passing only this CSV's
# ids would clobber rows registered by `_01` or `_02` for the same scope.
# Union with existing ids — the write is idempotent and self-heals partial runs.
try:
    existing_assoc_ids = set(
        pl.read_delta(OUTPUT_ROOT + "dataitem_dataset_association/")
          .filter((pl.col("project_id") == PROJECT_ID) & (pl.col("dataset_id") == DATASET_ID))
          ["dataitem_id"].to_list()
    )
except Exception:
    existing_assoc_ids = set()
full_assoc_ids = sorted(existing_assoc_ids | set(cell_ids))
associations = [
    DataItemDataSetAssociation(
        dataitem_id=cid, dataset_id=DATASET_ID, project_id=PROJECT_ID,
    )
    for cid in full_assoc_ids
]
result = write_models(associations, output_root=OUTPUT_ROOT)
print(f"DataItemDataSetAssociation written: {result.rows_written} rows")

DataItemDataSetAssociation written: 345 rows


In [10]:
assoc_verify = (
    pl.read_delta(OUTPUT_ROOT + "dataitem_dataset_association/")
    .filter((pl.col("project_id") == PROJECT_ID) & (pl.col("dataset_id") == DATASET_ID))
)
print(assoc_verify.shape)
print(assoc_verify.head(3))
assert assoc_verify.shape[0] == len(cell_ids), (
    f"Expected {len(cell_ids)} associations, got {assoc_verify.shape[0]}"
)
assert set(assoc_verify["dataitem_id"].to_list()) == set(cell_ids), "Association cell ids do not match CSV"
assert (assoc_verify["dataset_id"] == DATASET_ID).all(), "Not all associations point to DATASET_ID" 

(345, 3)
shape: (3, 3)
┌─────────────────────────────┬──────────────┬────────────┐
│ dataitem_id                 ┆ dataset_id   ┆ project_id │
│ ---                         ┆ ---          ┆ ---        │
│ str                         ┆ str          ┆ str        │
╞═════════════════════════════╪══════════════╪════════════╡
│ 17109_6201-X4328-Y6753_reg  ┆ visp_exc_wnm ┆ visp_wnm   │
│ 17109_6301-X4756-Y24516_reg ┆ visp_exc_wnm ┆ visp_wnm   │
│ 17109_6601-X4384-Y7436_reg  ┆ visp_exc_wnm ┆ visp_wnm   │
└─────────────────────────────┴──────────────┴────────────┘


## Write ipsilateral wide parquet (`projectionmeasurementmatrix/wnm_exc_proj_ipsi/`)

In [11]:
# Wide-form table: id (cell), project_id, dataset_id, then one float64 column per ipsi region acronym.
# Column order: id, project_id, dataset_id, then acronyms in CSV column order (after prefix strip).
ipsi_wide = df[ipsi_cols].copy()
ipsi_wide.columns = ipsi_acronyms
ipsi_wide = ipsi_wide.reset_index()  # adds 'id' column from the index
ipsi_wide.insert(1, "project_id", PROJECT_ID)
ipsi_wide.insert(2, "dataset_id", DATASET_ID)

# Build an explicit pyarrow schema so column types are pinned (id strings, values float64).
ipsi_schema = pa.schema(
    [pa.field("id", pa.string(), nullable=False),
     pa.field("project_id", pa.string(), nullable=False),
     pa.field("dataset_id", pa.string(), nullable=False)]
    + [pa.field(a, pa.float64()) for a in ipsi_acronyms]
)
ipsi_table = pa.Table.from_pandas(ipsi_wide, schema=ipsi_schema, preserve_index=False)

write_deltalake(
    OUTPUT_ROOT + f"projectionmeasurementmatrix/{FSI_IPSI}/", ipsi_table,
    mode="overwrite",
    predicate=f"project_id = '{PROJECT_ID}'",
    partition_by=["project_id"],
)
print("Ipsi wide parquet written:", ipsi_table.shape)

Ipsi wide parquet written: (345, 155)


In [12]:
ipsi_verify = (
    pl.read_delta(OUTPUT_ROOT + f"projectionmeasurementmatrix/{FSI_IPSI}/")
    .filter(pl.col("project_id") == PROJECT_ID)
)
print(ipsi_verify.shape)
print(ipsi_verify.select(["id", "project_id", "dataset_id"] + ipsi_acronyms[:3]).head(3))
assert ipsi_verify.shape == (len(cell_ids), 3 + len(ipsi_acronyms)), (
    f"Expected {(len(cell_ids), 3 + len(ipsi_acronyms))}, got {ipsi_verify.shape}"
)
assert set(ipsi_verify["id"].to_list()) == set(cell_ids), "Ipsi parquet cell ids do not match CSV"
assert (ipsi_verify["dataset_id"] == DATASET_ID).all(), "Not all rows have correct dataset_id" 

(345, 155)
shape: (3, 6)
┌─────────────────────────────┬────────────┬──────────────┬────────────┬──────────────┬────────────┐
│ id                          ┆ project_id ┆ dataset_id   ┆ VISam      ┆ VISp         ┆ VISpm      │
│ ---                         ┆ ---        ┆ ---          ┆ ---        ┆ ---          ┆ ---        │
│ str                         ┆ str        ┆ str          ┆ f64        ┆ f64          ┆ f64        │
╞═════════════════════════════╪════════════╪══════════════╪════════════╪══════════════╪════════════╡
│ 18864_6734-X4899-Y27447_reg ┆ visp_wnm   ┆ visp_exc_wnm ┆ 8287.70664 ┆ 34450.175934 ┆ 483.223644 │
│ 191812_7938-X6892-Y25312_re ┆ visp_wnm   ┆ visp_exc_wnm ┆ 0.0        ┆ 794.102517   ┆ 0.0        │
│ g                           ┆            ┆              ┆            ┆              ┆            │
│ 211550_7718-X19461-Y16950_r ┆ visp_wnm   ┆ visp_exc_wnm ┆ 0.0        ┆ 6473.751624  ┆ 0.0        │
│ eg                          ┆            ┆              ┆       

## Write contralateral wide parquet (`projectionmeasurementmatrix/wnm_exc_proj_contra/`)

In [13]:
contra_wide = df[contra_cols].copy()
contra_wide.columns = contra_acronyms
contra_wide = contra_wide.reset_index()
contra_wide.insert(1, "project_id", PROJECT_ID)
contra_wide.insert(2, "dataset_id", DATASET_ID)

contra_schema = pa.schema(
    [pa.field("id", pa.string(), nullable=False),
     pa.field("project_id", pa.string(), nullable=False),
     pa.field("dataset_id", pa.string(), nullable=False)]
    + [pa.field(a, pa.float64()) for a in contra_acronyms]
)
contra_table = pa.Table.from_pandas(contra_wide, schema=contra_schema, preserve_index=False)

write_deltalake(
    OUTPUT_ROOT + f"projectionmeasurementmatrix/{FSI_CONTRA}/", contra_table,
    mode="overwrite",
    predicate=f"project_id = '{PROJECT_ID}'",
    partition_by=["project_id"],
)
print("Contra wide parquet written:", contra_table.shape)

Contra wide parquet written: (345, 71)


In [14]:
contra_verify = (
    pl.read_delta(OUTPUT_ROOT + f"projectionmeasurementmatrix/{FSI_CONTRA}/")
    .filter(pl.col("project_id") == PROJECT_ID)
)
print(contra_verify.shape)
print(contra_verify.select(["id", "project_id", "dataset_id"] + contra_acronyms[:3]).head(3))
assert contra_verify.shape == (len(cell_ids), 3 + len(contra_acronyms)), (
    f"Expected {(len(cell_ids), 3 + len(contra_acronyms))}, got {contra_verify.shape}"
)
assert set(contra_verify["id"].to_list()) == set(cell_ids), "Contra parquet cell ids do not match CSV"
assert (contra_verify["dataset_id"] == DATASET_ID).all(), "Not all rows have correct dataset_id" 

(345, 71)
shape: (3, 6)
┌───────────────────────────────┬────────────┬──────────────┬─────────────┬──────┬─────┐
│ id                            ┆ project_id ┆ dataset_id   ┆ VISpor      ┆ VISp ┆ CP  │
│ ---                           ┆ ---        ┆ ---          ┆ ---         ┆ ---  ┆ --- │
│ str                           ┆ str        ┆ str          ┆ f64         ┆ f64  ┆ f64 │
╞═══════════════════════════════╪════════════╪══════════════╪═════════════╪══════╪═════╡
│ 18864_6734-X4899-Y27447_reg   ┆ visp_wnm   ┆ visp_exc_wnm ┆ 0.0         ┆ 0.0  ┆ 0.0 │
│ 191812_7938-X6892-Y25312_reg  ┆ visp_wnm   ┆ visp_exc_wnm ┆ 1045.437572 ┆ 0.0  ┆ 0.0 │
│ 211550_7718-X19461-Y16950_reg ┆ visp_wnm   ┆ visp_exc_wnm ┆ 0.0         ┆ 0.0  ┆ 0.0 │
└───────────────────────────────┴────────────┴──────────────┴─────────────┴──────┴─────┘


## Write `ProjectionMeasurementMatrix` metadata rows (one per laterality)

In [15]:
# region_coverage = subset of region_index with at least one non-zero value across all cells.
ipsi_nonzero_mask   = (df[ipsi_cols].fillna(0) != 0).any(axis=0)
contra_nonzero_mask = (df[contra_cols].fillna(0) != 0).any(axis=0)
ipsi_coverage   = [a for a, col in zip(ipsi_acronyms, ipsi_cols)     if bool(ipsi_nonzero_mask[col])]
contra_coverage = [a for a, col in zip(contra_acronyms, contra_cols) if bool(contra_nonzero_mask[col])]
print(f"ipsi region_coverage  : {len(ipsi_coverage)} / {len(ipsi_acronyms)}")
print(f"contra region_coverage: {len(contra_coverage)} / {len(contra_acronyms)}")

ipsi region_coverage  : 152 / 152
contra region_coverage: 66 / 68


In [16]:
# Use Path.resolve() to build absolute file:// URIs for the values pointer.
ipsi_values_uri   = f"file://{Path(OUTPUT_ROOT).resolve()}/projectionmeasurementmatrix/{FSI_IPSI}/"
contra_values_uri = f"file://{Path(OUTPUT_ROOT).resolve()}/projectionmeasurementmatrix/{FSI_CONTRA}/"

ipsi_matrix = ProjectionMeasurementMatrix(
    id=FSI_IPSI,
    description="WNM excitatory ipsilateral projection matrix: per-cell axon length (µm, inferred) by ipsilateral CCF region.",
    measurement_type=ProjectionMeasurementType.MICRONS_OF_AXON,
    modality=Modality.MORPHOLOGY,
    laterality=Laterality.IPSILATERAL,
    region_index=ipsi_acronyms,
    region_coverage=ipsi_coverage,
    data_item_index=cell_ids,
    values=ipsi_values_uri,
    unit=Unit.MICRONS_LENGTH,
)
contra_matrix = ProjectionMeasurementMatrix(
    id=FSI_CONTRA,
    description="WNM excitatory contralateral projection matrix: per-cell axon length (µm, inferred) by contralateral CCF region.",
    measurement_type=ProjectionMeasurementType.MICRONS_OF_AXON,
    modality=Modality.MORPHOLOGY,
    laterality=Laterality.CONTRALATERAL,
    region_index=contra_acronyms,
    region_coverage=contra_coverage,
    data_item_index=cell_ids,
    values=contra_values_uri,
    unit=Unit.MICRONS_LENGTH,
)

write_projection_matrix(ipsi_matrix, df[ipsi_cols].to_numpy(), output_root=OUTPUT_ROOT)
write_projection_matrix(contra_matrix, df[contra_cols].to_numpy(), output_root=OUTPUT_ROOT)
print("ProjectionMeasurementMatrix written: 2 rows")


ProjectionMeasurementMatrix written: 2 rows


In [17]:
pmm_verify = (
    pl.read_delta(OUTPUT_ROOT + "projectionmeasurementmatrix/")
    .filter(pl.col("id").is_in([FSI_IPSI, FSI_CONTRA]))
    .sort("id")
)
print(pmm_verify.shape)
print(pmm_verify.select(["id", "laterality", "measurement_type", "unit", "values"]))
assert pmm_verify.shape[0] == 2, f"Expected 2 ProjectionMeasurementMatrix rows, got {pmm_verify.shape[0]}"
assert set(pmm_verify["id"].to_list()) == {FSI_IPSI, FSI_CONTRA}
laterality_by_id = dict(zip(pmm_verify["id"].to_list(), pmm_verify["laterality"].to_list()))
assert laterality_by_id[FSI_IPSI]   == Laterality.IPSILATERAL.value
assert laterality_by_id[FSI_CONTRA] == Laterality.CONTRALATERAL.value
# data_item_index length matches row count of the wide parquets.
for row in pmm_verify.iter_rows(named=True):
    assert len(row["data_item_index"]) == len(cell_ids), (
        f"data_item_index length mismatch for {row['id']}: {len(row['data_item_index'])} vs {len(cell_ids)}"
    )
print("Verified both matrix rows.")

(2, 10)
shape: (2, 5)
┌─────────────────────┬───────────────┬──────────────────┬────────────────┬────────────────────────┐
│ id                  ┆ laterality    ┆ measurement_type ┆ unit           ┆ values                 │
│ ---                 ┆ ---           ┆ ---              ┆ ---            ┆ ---                    │
│ str                 ┆ str           ┆ str              ┆ str            ┆ str                    │
╞═════════════════════╪═══════════════╪══════════════════╪════════════════╪════════════════════════╡
│ wnm_exc_proj_contra ┆ CONTRALATERAL ┆ MICRONS_OF_AXON  ┆ MICRONS_LENGTH ┆ file:///scratch/em_pat │
│                     ┆               ┆                  ┆                ┆ chseq_wn…              │
│ wnm_exc_proj_ipsi   ┆ IPSILATERAL   ┆ MICRONS_OF_AXON  ┆ MICRONS_LENGTH ┆ file:///scratch/em_pat │
│                     ┆               ┆                  ┆                ┆ chseq_wn…              │
└─────────────────────┴───────────────┴──────────────────┴───────────

## Summary

| Output path | Class | Rows |
|---|---|---|
| `dataitem/` | `DataItem` | +N new cells (4 expected; via `append_new_dataitems`) |
| `dataitem_dataset_association/` | `DataItemDataSetAssociation` | full scope = existing ∪ 345 cells (overwrite_scoped on `(project_id, dataset_id)`; union preserves rows from `_01`/`_02`) |
| `projectionmeasurementmatrix/wnm_exc_proj_ipsi/` | wide parquet | 345 cells × 152 ipsilateral region columns |
| `projectionmeasurementmatrix/wnm_exc_proj_contra/` | wide parquet | 345 cells × 68 contralateral region columns |
| `projectionmeasurementmatrix/` | `ProjectionMeasurementMatrix` | 2 (one per laterality) |

`measurement_type=MICRONS_OF_AXON` is recorded based on inference from value magnitudes; awaiting confirmation from the data owner. Region indices are stored as raw acronym strings until `brainregion/` is bootstrapped (see schema-mismatch note above).
